# Fast EDA Notebook

This notebook is intentionally fast. It samples the large JSONL file instead of loading all 100,000 candidates into memory.

In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

print("Project root:", PROJECT_ROOT)
print("Data files:")
for path in sorted(DATA_DIR.iterdir()):
    print("-", path.name)

## Fast Candidate Sample

Do not use `pd.read_json("data/candidates.jsonl", lines=True)` for EDA. The file is large, so this cell reads only the first 1,000 rows with the standard library.

In [ ]:
import json

def load_candidate_sample(n=1000):
    rows = []
    with (DATA_DIR / "candidates.jsonl").open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            rows.append(json.loads(line))
    return rows

candidates = load_candidate_sample(1000)
first = candidates[0]

print("Loaded sample rows:", len(candidates))
print("Columns:", list(first.keys()))
print("First candidate:", first["candidate_id"])

## Inspect One Candidate

In [ ]:
print(json.dumps(first["profile"], indent=2))
print("\nSkills:")
for skill in first["skills"][:10]:
    print("-", skill["name"], skill.get("proficiency", ""))
print("\nRedrob signals:")
print(json.dumps(first["redrob_signals"], indent=2))

## Read Job Description

In [ ]:
from docx import Document

doc = Document(DATA_DIR / "job_description.docx")
job_text = "\n".join(p.text for p in doc.paragraphs)

print("JD characters:", len(job_text))
print(job_text[:1200])

## View Final Submission

The full ranking should be generated in `src/ranker.py` or saved output, not recomputed inside EDA cells.

In [ ]:
import csv

submission_path = OUTPUT_DIR / "submission.csv"

with submission_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    rows = [row for _, row in zip(range(10), reader)]

for row in rows:
    print(row["rank"], row["candidate_id"], row["score"], row.get("reasoning", ""))